Phase 5: GNN edge classifier (GINE, plain PyTorch). One notebook, two modes, set by DEBUG in cell 2.

What it does: every transaction becomes an edge between two accounts. The model scores each transaction from the graph of
all transactions on EARLIER DAYS (a daily snapshot, like the daily PageRank), plus the transaction's own features.
Nothing from the transaction's own day or later is ever visible. Day-1 transactions are not trained on (empty snapshot).

Run it in this order:
  1. DEBUG = True, accelerator None (CPU): a 5% random sample; checks the pipeline, the no-future-edge assertion, and that
     the model can overfit a small batch. Takes a few minutes. Fix anything that fails before step 2.
  2. DEBUG = False, accelerator GPU T4 x1, Save Version -> Save & Run All (Commit). Expect roughly 1 to 2 hours.

Before you start: push the latest code to GitHub (papertrail.gnn is new), add the papertrail-processed dataset as input,
Internet ON, attach the DAGSHUB_TOKEN secret. PyTorch is preinstalled on Kaggle; no torch-geometric/pyg-lib needed.

Reference to beat (test split, 3 seeds, stable XGBoost settings, notebook 03d): XGB tabular PR-AUC 0.065,
XGB with history features PR-AUC 0.442 +- 0.004 (F1 0.499, recall@1%FPR 0.675). Report the GNN against THAT, not against 0.046.



In [ ]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git
import os
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
import dagshub, mlflow
dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)


In [ ]:
import glob
import json
import time

import numpy as np
import polars as pl
import torch

from papertrail.gnn.data import TEST, TRAIN, VAL, Snapshot, build_edge_table, make_batch
from papertrail.gnn.train import GNNConfig, SnapshotCache, overfit_check, predict, train_gnn
from papertrail.metrics import best_f1_threshold, evaluate
from papertrail.split import assert_valid_split, split_summary, temporal_split

DEBUG = True            # True: 5% sample on CPU, a few epochs.  False: full data, GPU, full training.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| DEBUG =", DEBUG)

if DEBUG:
    cfg = GNNConfig(hidden=32, batch_size=256, steps_per_epoch=40, max_epochs=3, patience=3, eval_batch=1024)
else:
    cfg = GNNConfig()    # hidden 64, 2 layers, fanouts (10, 5), 500 steps/epoch, up to 30 epochs, patience 5
print(cfg)


In [ ]:
df = pl.read_parquet(glob.glob("/kaggle/input/**/transactions.parquet", recursive=True)[0]).sort(["ts", "txn_id"])
if DEBUG:
    df = df.sample(fraction=0.05, seed=0).sort(["ts", "txn_id"])        # random 5%: structure is thin, only the plumbing matters
summary = split_summary(df)
print(summary)
assert_valid_split(summary)
train, val, test = temporal_split(df)

t0 = time.time()
table = build_edge_table(train, val, test)
all_txn_id = pl.concat([train, val, test])["txn_id"].to_numpy()
print(f"edge table in {time.time() - t0:.0f}s: {table.n_edges} edges, {table.n_nodes} accounts, "
      f"{len(table.feature_names)} edge features, positives {int(table.label.sum())}")
print("edge features:", table.feature_names)
print("days:", {int(d): int((table.day == d).sum()) for d in np.unique(table.day)})


In [ ]:
snaps = SnapshotCache(table)
rng = np.random.default_rng(0)
for split, name in ((TRAIN, "train"), (VAL, "val"), (TEST, "test")):
    ids = table.ids(split, min_day=2)
    day = int(table.day[ids].min())
    snap = snaps[day]
    sub = rng.choice(ids[table.day[ids] == day], size=min(512, int((table.day[ids] == day).sum())), replace=False)
    b = make_batch(table, snap, sub, cfg.fanouts, rng)
    used = b["edge_global"] % max(snap.n, 1)                             # which transaction each sampled edge is
    assert (table.day[used] < day).all(), "a sampled neighbour is not from an earlier day"
    assert not np.isin(sub, used).any(), "a target edge appears among its own neighbours"
    print(f"{name:5s} day {day}: snapshot has {snap.n} transactions; batch of {len(sub)} targets -> "
          f"{len(b['nodes'])} nodes, {b['edge_index'].shape[1]} edges; latest neighbour day {int(table.day[used].max()) if len(used) else '-'}")
print("no-future-edge check passed")


In [ ]:
final_loss = overfit_check(table, cfg, device, n=256, steps=300)
assert final_loss < 0.1, f"could not overfit a tiny batch (final loss {final_loss:.3f}); the pipeline has a bug"
print("overfit check passed")


In [ ]:
val_ids, test_ids = table.ids(VAL), table.ids(TEST)
t0 = time.time()
mlflow.start_run(run_name="gnn_gine" + ("_debug" if DEBUG else ""))
mlflow.log_params({**cfg.as_dict(), "debug": DEBUG, "n_edges": table.n_edges, "n_edge_features": len(table.feature_names)})
model, history, snaps = train_gnn(table, cfg, device, val_ids, snaps=snaps)
for h in history:
    mlflow.log_metrics({"epoch_train_loss": h["train_loss"], "epoch_val_pr_auc": h["val_pr_auc"]}, step=h["epoch"])
print(f"training took {time.time() - t0:.0f}s over {len(history)} epochs")


In [ ]:
p_val = predict(model, table, val_ids, cfg, device, snaps)
p_test = predict(model, table, test_ids, cfg, device, snaps)
y_val, y_test = table.label[val_ids], table.label[test_ids]
thr = best_f1_threshold(y_val, p_val)
res = {"threshold": float(thr)}
for k, v in evaluate(y_val, p_val, thr).items():
    res[f"val_{k}"] = v
for k, v in evaluate(y_test, p_test, thr).items():
    res[f"test_{k}"] = v
test_df = pl.concat([train, val, test])[test_ids]
ach = (test_df["pay_format"] == "ACH").to_numpy()
for k, v in evaluate(y_test[ach], p_test[ach], thr).items():
    res[f"test_ach_{k}"] = v
mlflow.log_metrics(res)
mlflow.end_run()

print({k: round(v, 4) for k, v in res.items() if k.startswith("test_") and "ach" not in k})
print("reference (03d, test): XGB history features PR-AUC 0.442 / F1 0.499 / recall@1%FPR 0.675 / precision@1000 0.461")


In [ ]:
suffix = "_debug" if DEBUG else ""
torch.save(model.state_dict(), f"/kaggle/working/gnn_gine{suffix}.pt")
scores = pl.DataFrame({
    "txn_id": np.concatenate([all_txn_id[val_ids], all_txn_id[test_ids]]),
    "split": ["val"] * len(val_ids) + ["test"] * len(test_ids),
    "gnn_score": np.concatenate([p_val, p_test]),
    "label": np.concatenate([y_val, y_test]).astype(int),
})
scores.write_parquet(f"/kaggle/working/gnn_scores{suffix}.parquet")
with open(f"/kaggle/working/gnn_metrics{suffix}.json", "w") as fh:
    json.dump({"config": cfg.as_dict(), "history": history, "metrics": res, "debug": DEBUG}, fh, indent=2)
print("Download gnn_metrics*.json (and gnn_scores*.parquet for the blend step) from the Output tab into PaperTrail/reports/")
